In [ ]:
# --- IMPORTS ---
import os
import re
import subprocess
from pathlib import Path
from typing import Dict, Any, Tuple, List

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# --- CONSTANTS ---
CLK_STD_NS = 10.0
CLK_HP_NS  = 5.0
VIVADO_EXE = r"D:\Softwares\AMD-Design\2025.2\Vivado\bin\vivado.bat"

VARIANTS = [
    "baseline",
    "registering",
    "reordering",
    "registering_reordering",
    "isolated_reordering"
]

PROJECT_ROOT       = Path("..").resolve()
SCRIPTS_FOLDER     = PROJECT_ROOT / "scripts" / "project-03"
OUTPUT_SIM_FOLDER  = PROJECT_ROOT / "notebooks" / "output" / "project-03" / "sims"
REPORTS_FOLDER     = OUTPUT_SIM_FOLDER / "reports"

In [ ]:
# --- PROJECT INITIALIZATION ---
pd.set_option('display.float_format', '{:.4f}'.format)
OUTPUT_SIM_FOLDER.mkdir(parents=True, exist_ok=True)

all_metrics: List[Dict[str, Any]] = []

def run_vivado_variant(variant: str, clk_period_ns: float) -> None:
    """Executes the TCL run script for a given architectural variant and clock constraint."""
    working_dir = OUTPUT_SIM_FOLDER / variant
    report_dir = REPORTS_FOLDER / variant
    working_dir.mkdir(parents=True, exist_ok=True)
    report_dir.mkdir(parents=True, exist_ok=True)
    log_file = report_dir / f"vivado_console_{clk_period_ns}ns.log"

    # 1. Sanitize the environment
    # Jupyter and Python inject variables that break Vivado's internal Tcl/Python interpreters
    clean_env = os.environ.copy()
    unsafe_keys = [
        "PYTHONHOME", "PYTHONPATH", "PYTHONEXECUTABLE",
        "TCL_LIBRARY", "TK_LIBRARY", "JUPYTER_PATH",
        "CONDA_PREFIX", "CONDA_DEFAULT_ENV", "VIRTUAL_ENV"
    ]
    for key in unsafe_keys:
        if key in clean_env:
            del clean_env[key]

    # Force the correct command interpreter
    clean_env["COMSPEC"] = r"C:\Windows\System32\cmd.exe"

    # 2. Create a temporary batch file to emulate a fresh command prompt execution
    tcl_script = SCRIPTS_FOLDER / "run_variant.tcl"
    bat_content = f"""@echo off
REM Launch Vivado in batch mode
"{VIVADO_EXE}" -mode batch -log "{log_file}" -source "{tcl_script}" -tclargs {variant} {clk_period_ns}
exit /b %errorlevel%
"""

    bat_file_path = OUTPUT_SIM_FOLDER / f"launch_{variant}_{clk_period_ns}ns.bat"

    with open(bat_file_path, "w") as bat_file:
        bat_file.write(bat_content)

    print(f"Running Vivado for variant '{variant}' at {clk_period_ns}ns...")

    # 3. Execute the wrapper script
    cmd = [str(bat_file_path)]

    try:
        subprocess.run(cmd, check=True, shell=False, cwd=str(working_dir), env=clean_env)
    finally:
        # Cleanup the temporary batch file after execution
        if bat_file_path.exists():
            try:
                bat_file_path.unlink()
            except PermissionError:
                pass # Ignore cleanup errors if file is locked

def parse_utilization(report_path: Path) -> Tuple[float, float]:
    """Extracts LUT and FF usage from the Vivado utilization report."""
    lut, ff = np.nan, np.nan
    if not report_path.exists():
        return lut, ff

    with open(report_path, 'r') as f:
        content = f.read()
        lut_match = re.search(r'\|\s*Slice LUTs\s*\|\s*(\d+)', content)
        ff_match = re.search(r'\|\s*Slice Registers\s*\|\s*(\d+)', content)
        if lut_match:
            lut = float(lut_match.group(1))
        if ff_match:
            ff = float(ff_match.group(1))

    return lut, ff

def parse_timing_summary(report_path: Path) -> float:
    """Extracts Worst Negative Slack (WNS) from the Vivado timing report."""
    wns = np.nan
    if not report_path.exists():
        return wns

    with open(report_path, 'r') as f:
        content = f.read()
        wns_match = re.search(r'WNS\(ns\)[^\n]*\n[-]+\n\s*([-\d\.]+)', content)
        if wns_match:
            wns = float(wns_match.group(1))

    return wns

def parse_power(report_path: Path) -> Tuple[float, float, float, float]:
    """Extracts Logic, Signals, Clock, and Total Internal power metrics."""
    logic, signals, clock, total_int = np.nan, np.nan, np.nan, np.nan
    if not report_path.exists():
        return logic, signals, clock, total_int

    with open(report_path, 'r') as f:
        content = f.read()
        logic_match = re.search(r'\|\s*Logic\s*\|\s*([\d\.]+)', content)
        sig_match = re.search(r'\|\s*Signals\s*\|\s*([\d\.]+)', content)
        clk_match = re.search(r'\|\s*Clocks\s*\|\s*([\d\.]+)', content)

        if logic_match: logic = float(logic_match.group(1))
        if sig_match: signals = float(sig_match.group(1))
        if clk_match: clock = float(clk_match.group(1))

        total_int = np.nansum([logic, signals, clock])

    return logic, signals, clock, total_int

def extract_metrics(variant: str, clk_period_ns: float) -> Dict[str, Any]:
    """Consolidates parsed metrics into a dictionary for a specific variant build."""
    report_dir = REPORTS_FOLDER / variant

    lut, ff = parse_utilization(report_dir / "utilization.rpt")
    wns = parse_timing_summary(report_dir / "timing_summary.rpt")
    fmax = 1000.0 / (clk_period_ns - wns) if not np.isnan(wns) else np.nan
    logic, signals, clock, total_int = parse_power(report_dir / "power.rpt")

    return {
        "Variant": variant,
        "Clock Constraint [ns]": clk_period_ns,
        "LUT": lut,
        "FF": ff,
        "WNS [ns]": wns,
        "fmax [MHz]": fmax,
        "Logic Power [W]": logic,
        "Signals Power [W]": signals,
        "Clock Power [W]": clock,
        "Total Internal Power [W]": total_int
    }

## STEP 1 & 2: Baseline RTL Implementation, Synthesis and Power Characterization
I implement the **reference circuit** in `VHDL` and run the **Baseline** variant through synthesis and implementation in **Vivado**, targeting a standard clock constraint of `10ns`.

From the Vivado reports, I extract the resource usage (`LUT`, `FF`), timing metrics (`WNS`, `fmax`), and dynamic power breakdown (`Logic`, `Signals`, `Clock`). These act as the comparison anchor for every optimized variant.

In [ ]:
# --- BASELINE METRICS EXTRACTION ---
variant_baseline = "baseline"

run_vivado_variant(variant_baseline, CLK_STD_NS)

baseline_metrics = extract_metrics(variant_baseline, CLK_STD_NS)
all_metrics.append(baseline_metrics)

df_baseline = pd.DataFrame([baseline_metrics])
display(df_baseline)

## STEP 3: Registering Technique
I implement the **Registering** variant by inserting an intermediate pipeline stage for the operands and selection signal to prevent glitches from reaching the adder input.

The Vivado flow is executed under the `10ns` constraint. The dynamic power reduction should stem from a lower switching factor $\alpha$, trading off against increased `FF` count and clock-network power.

In [ ]:
# --- REGISTERING METRICS EXTRACTION ---
variant_registering = "registering"

run_vivado_variant(variant_registering, CLK_STD_NS)

registering_metrics = extract_metrics(variant_registering, CLK_STD_NS)
all_metrics.append(registering_metrics)

df_registering = pd.DataFrame([registering_metrics])
display(df_registering)

## STEP 4: Reordering Technique
I implement the **Reordering** variant by replacing the shared adder with two dedicated 32-bit adders operating in parallel. A final `MUX_SUM` selects the correct sum, removing the pre-adder multiplexer.

Metrics are extracted post-implementation under the `10ns` clock to observe the trade-off: increased `LUT` usage against reduced switching activity on the sum path.

In [ ]:
# --- REORDERING METRICS EXTRACTION ---
variant_reordering = "reordering"

run_vivado_variant(variant_reordering, CLK_STD_NS)

reordering_metrics = extract_metrics(variant_reordering, CLK_STD_NS)
all_metrics.append(reordering_metrics)

df_reordering = pd.DataFrame([reordering_metrics])
display(df_reordering)

## STEP 5: Combined Registering & Reordering
I implement the **Registering & Reordering** variant to combine both techniques. The pipeline stage encloses the two dedicated adders, removing glitches on the `MUX_SUM` select line.

Metrics are extracted at `10ns` to assess the combined timing and power benefits.

In [ ]:
# --- COMBINED REGISTERING & REORDERING METRICS EXTRACTION ---
variant_comb = "registering_reordering"

run_vivado_variant(variant_comb, CLK_STD_NS)

comb_metrics = extract_metrics(variant_comb, CLK_STD_NS)
all_metrics.append(comb_metrics)

df_comb = pd.DataFrame([comb_metrics])
display(df_comb)

## STEP 6: Gated Inputs, Precomputing & Clock Gating (Isolated Reordering)
I implement the **Isolated Reordering** variant on top of the Reordering topology. It leverages **Precomputing**, **Gated Inputs**, and **Clock Gating** (via `CE` pins) to eliminate redundant switching in the unselected adder branch without the heavy logic cost.

Metrics are extracted under the `10ns` clock constraint to verify logic and signal power savings.

In [ ]:
# --- ISOLATED REORDERING METRICS EXTRACTION ---
variant_isolated = "isolated_reordering"

run_vivado_variant(variant_isolated, CLK_STD_NS)

isolated_metrics = extract_metrics(variant_isolated, CLK_STD_NS)
all_metrics.append(isolated_metrics)

df_isolated = pd.DataFrame([isolated_metrics])
display(df_isolated)

## STEP 8: Cross-Configuration Comparison
For validating the reproduced methodology, a high-performance clock campaign (`5ns`) is executed across all configurations.

The extracted metrics from both campaigns (`10ns` and `5ns`) are consolidated to highlight:
- **Resource usage trade-off**: `LUT`/`FF` overhead.
- **Timing improvement**: `WNS` margin and `fmax` gain.
- **Dynamic power reduction**: Decreases in internal power consumption.

In [ ]:
# --- CROSS-CONFIGURATION COMPARISON ---

for variant in VARIANTS:
    run_vivado_variant(variant, CLK_HP_NS)
    hp_metrics = extract_metrics(variant, CLK_HP_NS)
    all_metrics.append(hp_metrics)

df_comparison = pd.DataFrame(all_metrics)

print("=" * 100)
print("CROSS-CONFIGURATION COMPARISON SUMMARY")
print("=" * 100)
display(df_comparison.sort_values(by=["Clock Constraint [ns]", "Variant"]))

df_10ns = df_comparison[df_comparison["Clock Constraint [ns]"] == CLK_STD_NS].set_index("Variant")

fig, ax = plt.subplots(figsize=(10, 6))
df_10ns[["Logic Power [W]", "Signals Power [W]", "Clock Power [W]"]].plot(kind="bar", stacked=True, ax=ax, colormap="viridis")

ax.set_title("Dynamic Power Breakdown at 10ns Clock", fontsize=14)
ax.set_ylabel("Power [W]", fontsize=12)
ax.set_xlabel("Architectural Variant", fontsize=12)
ax.grid(axis='y', linestyle='--', alpha=0.7)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()

plt.show()